In [0]:
from pyspark.sql.functions import current_timestamp, col

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"
target_table = f"{catalog}.{schema}.finnhub_news_bronze"
checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume}/_state/checkpoints/finnhub_news_bronze"
schema_location = f"/Volumes/{catalog}/{schema}/{volume}/_state/schemas/finnhub_news_bronze"

In [0]:
# Auto Loader Stream Ingestion
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("multiLine", "true")
    .load(landing_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", current_timestamp())
)

query = (
    df_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
# print(f"Successfully processed Auto Loader batch into Bronze table: {target_table}")

In [0]:
# df_verify_news = spark.table(f"{catalog}.{schema}.finnhub_news_bronze")

# print(f"Total Bronze News Articles: {df_verify_news.count()}")
# print("Schema:")
# df_verify_news.printSchema()
# display(df_verify_news.limit(5))

In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# target_table = f"{catalog}.{schema}.finnhub_news_bronze"

# df = spark.read.table(target_table)

# print("--- Schema Evolution Breakdown by Phase ---")
# display(
#     df.groupBy("_schema_phase")
#     .count()
# )

# print("--- Sample showing evolved field 'index_tracker' ---")
# display(df.select("id", "_schema_phase", "index_tracker", "related", "headline").limit(10))